In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# --- 1. Load Your Pre-computed Data ---
# Load the embeddings you saved from your previous notebook.
print("--- 1. Loading pre-computed embeddings and post data ---")
try:
    embeddings_data = np.load('showerthoughts_embeddings.npz')
    # It's good practice to allow loading of pickled objects if needed, but be aware of security.
    # For this project, it's safe as you created the file.
    # embeddings_data = np.load('post_embeddings.npz', allow_pickle=True)
except FileNotFoundError:
    print("Error: 'showerthoughts_embeddings.npz' not found. Please run the embedding generation script first.")
    exit()

# Load the original dataframe to get post titles for presenting results.
try:
    posts_df = pd.read_csv('reddit_showerthoughts.tsv', sep='\t', on_bad_lines='skip')
    # Ensure the dataframe is clean, especially the title column
    posts_df = posts_df.dropna(subset=['title'])
    posts_df = posts_df.reset_index(drop=True)
except FileNotFoundError:
    print("Error: 'reddit_showerthoughts.tsv' not found.")
    exit()

# Extract the arrays from the loaded file
post_ids = embeddings_data['ids']
title_embeddings = embeddings_data['title_embed']
content_embeddings = embeddings_data['content_embed']

print(f"Loaded {len(post_ids)} post IDs and their embeddings.")
print("-" * 30)

--- 1. Loading pre-computed embeddings and post data ---
Loaded 95084 post IDs and their embeddings.
------------------------------


In [6]:
# --- 2. Combine Title and Selftext Embeddings ---
# A simple and effective way to combine them is to average them.
# This creates a single vector representing the overall meaning of the post.
print("\n--- 2. Combining title and selftext embeddings ---")
# We use a weighted average. Let's give the title slightly more importance.
# You can experiment with these weights.
title_weight = 0.8
content_weight = 0.2
combined_embeddings = (title_weight * title_embeddings) + (content_weight * content_embeddings)
print(f"Combined embeddings created with shape: {combined_embeddings.shape}")
print("-" * 30)


--- 2. Combining title and selftext embeddings ---
Combined embeddings created with shape: (95084, 768)
------------------------------


In [7]:
# --- 3. Simulate a User and Build a User Profile ---
# To test our recommender, let's pretend to be a user who liked a few specific posts.
# We will find these posts in our dataframe and use their embeddings to create a user profile.
print("\n--- 3. Simulating a user and building their profile ---")

# Let's pick a few posts based on their titles.
liked_post_titles = [
    "One day everyone just stopped wearing capris and no one talks about.",
    "100 years ago a three hundred pound fat guy was a side show in a circus, now hes just a guy.",
    "If you think your life sucks, chill. There are people who didn't like Game Of Thrones' ending."
]

# Find the indices of these posts in our dataframe
liked_indices = posts_df[posts_df['title'].isin(liked_post_titles)].index.tolist()

if not liked_indices:
    print("Could not find the example liked posts in the dataset. Picking 3 random posts instead.")
    liked_indices = np.random.choice(len(posts_df), 3, replace=False).tolist()

print("Simulated user liked the following posts:")
for idx in liked_indices:
    print(f"  - {posts_df.iloc[idx]['title']}")

# Build the user profile by averaging the embeddings of the posts they liked.
user_profile_vector = np.mean(combined_embeddings[liked_indices], axis=0)
# Reshape for sklearn's cosine_similarity function
user_profile_vector = user_profile_vector.reshape(1, -1)
print(f"\nUser profile vector created with shape: {user_profile_vector.shape}")
print("-" * 30)


--- 3. Simulating a user and building their profile ---
Simulated user liked the following posts:
  - If you think your life sucks, chill. There are people who didn't like Game Of Thrones' ending.
  - 100 years ago a three hundred pound fat guy was a side show in a circus, now hes just a guy.
  - One day everyone just stopped wearing capris and no one talks about.

User profile vector created with shape: (1, 768)
------------------------------


In [ ]:
# --- 4. Find and Rank Recommendations ---
# Now, we calculate the similarity between our user's profile and ALL posts in the dataset.
print("\n--- 4. Calculating similarity and finding recommendations ---")
# Use cosine similarity to find the most similar posts
similarity_scores = cosine_similarity(user_profile_vector, combined_embeddings)

# The result is a 2D array, so we flatten it to a 1D array of scores
similarity_scores = similarity_scores.flatten()

# Get the indices of the top N most similar posts
# We use argsort to get indices of sorted values, then reverse them for descending order.
N = 10
# We add len(liked_indices) to N because the most similar posts will be the ones the user already liked.
top_n_indices = similarity_scores.argsort()[-(N + len(liked_indices)) :][::-1]

# Filter out the posts the user has already seen
recommendation_indices = [idx for idx in top_n_indices if idx not in liked_indices]

print(f"\nTop {N} recommendations for our user:")
for i, idx in enumerate(recommendation_indices[:N]):
    # Get the post title from the original dataframe
    rec_title = posts_df.iloc[idx]['title']
    rec_score = similarity_scores[idx]
    print(f"{i+1}. (Score: {rec_score:.4f}) {rec_title}")